# Chandra2 parser-smoke benchmark

Notebook này chạy baseline `datalab-to/chandra-ocr-2` trực tiếp bằng Hugging Face trên GPU Colab, với prompt chuẩn `ocr_layout`. Input là đúng 5 file chung trong `data/raw/parser-smoke`; output của từng document được ghi vào Google Drive ngay sau khi document đó hoàn tất.

Yêu cầu: runtime GPU hỗ trợ BF16 và có ít nhất 18 GiB VRAM (ưu tiên L4/A100; G4/H100 cũng được nếu đạt điều kiện). Chọn **Runtime → Run all**.

In [2]:
import shutil
import subprocess
import sys

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Runtime hiện tại không có GPU. Hãy đổi Hardware accelerator sang GPU.")
gpu_status = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv,noheader",
    ],
    check=True,
    capture_output=True,
    text=True,
)
print(f"Python: {sys.version}")
print(f"GPU connected: {gpu_status.stdout.strip()}")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
GPU connected: NVIDIA L4, 23034 MiB, 22564 MiB, 580.82.07


In [3]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "chandra-ocr[hf]==0.2.0"]
)

0

In [4]:
from __future__ import annotations

import hashlib
import json
import os
import zipfile
from pathlib import Path


from google.colab import drive

EXPERIMENT_ID = "20260722T155731-parser-smoke-571c3721"
DRIVE_EXPERIMENT_DIR = (
    Path("/content/drive/MyDrive/AXIOM_DE-RD/parser-comparison") / EXPERIMENT_ID
)
CORPUS_ZIP = DRIVE_EXPERIMENT_DIR / "corpus.zip"
EXTRACT_DIR = Path("/content/parser-comparison") / EXPERIMENT_ID

drive.mount("/content/drive")
if not CORPUS_ZIP.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {CORPUS_ZIP}. Hãy upload corpus.zip vào đúng thư mục Drive."
    )

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CORPUS_ZIP) as archive:
    extract_root = EXTRACT_DIR.resolve()
    for member in archive.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if extract_root not in target.parents and target != extract_root:
            raise RuntimeError(f"ZIP member không an toàn: {member.filename}")
    archive.extractall(EXTRACT_DIR)

manifest = json.loads((EXTRACT_DIR / "manifest.json").read_text(encoding="utf-8"))
if manifest.get("experiment_id") != EXPERIMENT_ID:
    raise RuntimeError("experiment_id trong manifest không khớp notebook.")
if manifest.get("document_count") != 5:
    raise RuntimeError("Baseline phải có đúng 5 document.")

for document in manifest["documents"]:
    input_path = EXTRACT_DIR / "corpus" / document["document_id"] / document["filename"]
    actual_sha256 = hashlib.sha256(input_path.read_bytes()).hexdigest()
    if actual_sha256 != document["sha256"]:
        raise RuntimeError(f"Checksum không khớp: {document['relative_path']}")

print(f"Corpus OK: {len(manifest['documents'])} documents")
print(f"Expected pages: {manifest['expected_page_count']}")
print(f"Drive output: {DRIVE_EXPERIMENT_DIR / 'chandra2'}")

Mounted at /content/drive
Corpus OK: 5 documents
Expected pages: 6
Drive output: /content/drive/MyDrive/AXIOM_DE-RD/parser-comparison/20260722T155731-parser-smoke-571c3721/chandra2


In [10]:
import importlib.metadata

import torch

MODEL_CHECKPOINT = "datalab-to/chandra-ocr-2"
# PROMPT_NAME = "custom_ocr_layout_table_v2"
PROMPT_NAME = "custom_ocr_layout_table_merge_v2_short"
TARGET_DOCUMENT_IDS = {"95168ff83bfc14d8"}  # sample 5 only
BATCH_SIZE = 1
MAX_OUTPUT_TOKENS = 12384
INCLUDE_IMAGES = True
INCLUDE_HEADERS_FOOTERS = True

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch không nhận CUDA.")
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / (1024 ** 3)
if not torch.cuda.is_bf16_supported():
    raise RuntimeError(f"GPU {gpu.name} không hỗ trợ BF16. Hãy dùng L4/A100.")
if gpu_memory_gib < 18:
    raise RuntimeError(
        f"GPU {gpu.name} chỉ có {gpu_memory_gib:.1f} GiB VRAM; cần ít nhất 18 GiB."
    )

os.environ["MODEL_CHECKPOINT"] = MODEL_CHECKPOINT
os.environ["TORCH_DEVICE"] = "cuda"
os.environ.pop("TORCH_ATTN", None)

from chandra.input import load_file
from chandra.model import InferenceManager
from chandra.model.schema import BatchInputItem
from chandra.prompts import OCR_LAYOUT_PROMPT

TABLE_MERGE_RULES = """
Additional mandatory table-structure rules:

* Before generating a table, identify its complete logical row and column grid.
* Inspect horizontal and vertical borders carefully. A missing internal border usually means the cell continues across adjacent rows or columns.
* Use rowspan for vertical merged cells and colspan for horizontal merged cells.
* Count exactly how many logical rows or columns each merged cell covers.
* Emit a merged cell only once, at its upper-left logical grid position.
* Do not emit td/th elements for positions already covered by rowspan or colspan.
* Text visually centered across multiple rows belongs to the merged cell. Do not assign it only to the middle row.
* Do not duplicate merged-cell text into visually blank rows or columns.
* Preserve genuinely empty cells as empty cells.
* Ensure that expanding all rowspan and colspan values produces a rectangular, non-overlapping grid with the same number of logical columns in every row.
* Preserve the exact visible text. Do not invent values to fill blank cells.
* Tables must be returned as complete HTML table elements, including every row.
""".strip()

TABLE_MERGE_PROMPT_V2 = """
For every table, reconstruct the logical grid before writing HTML.

A missing internal horizontal border means the cell spans multiple rows; use rowspan.
A missing internal vertical border means the cell spans multiple columns; use colspan.

Place each merged cell at its top-left grid position and emit its content exactly once.
Do not create td/th elements for positions covered by a previous rowspan or colspan.
Do not assign vertically centered text to the middle row of a merged cell.
The expanded table grid must be rectangular and non-overlapping.
""".strip()

CUSTOM_PROMPT = OCR_LAYOUT_PROMPT + "\n\n" + TABLE_MERGE_PROMPT_V2

package_version = importlib.metadata.version("chandra-ocr")
prompt_hash = hashlib.sha256(CUSTOM_PROMPT.encode("utf-8")).hexdigest()
generation_config = {
    "model_checkpoint": MODEL_CHECKPOINT,
    "method": "hf",
    "prompt_type": PROMPT_NAME,
    "prompt_hash": prompt_hash,
    "target_document_ids": sorted(TARGET_DOCUMENT_IDS),
    "batch_size": BATCH_SIZE,
    "max_output_tokens": MAX_OUTPUT_TOKENS,
    "include_images": INCLUDE_IMAGES,
    "include_headers_footers": INCLUDE_HEADERS_FOOTERS,
    "package_version": package_version,
}
config_hash = hashlib.sha256(
    json.dumps(generation_config, sort_keys=True).encode("utf-8")
).hexdigest()
try:
    from huggingface_hub import model_info

    model_revision = model_info(MODEL_CHECKPOINT).sha
except Exception as exc:
    model_revision = None
    print(f"WARNING: không lấy được model revision: {exc}")

print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")
print(f"CUDA: {torch.version.cuda}; BF16: {torch.cuda.is_bf16_supported()}")
print(f"Chandra package: {package_version}; config: {config_hash[:12]}")
manager = InferenceManager(method="hf")
print("Chandra2 HF model loaded.")
subprocess.run(["nvidia-smi"], check=True)

GPU: NVIDIA L4 (22.0 GiB)
CUDA: 12.8; BF16: True
Chandra package: 0.2.0; config: 41b13ded5183


Loading weights:   0%|          | 0/724 [00:00<?, ?it/s]

Chandra2 HF model loaded.


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [12]:
import json
import shutil
import time
from collections import Counter
from datetime import datetime, timezone

# CHANDRA_ROOT = DRIVE_EXPERIMENT_DIR / "chandra2-table-prompt-v2"
CHANDRA_ROOT = (
    DRIVE_EXPERIMENT_DIR
    / "chandra2-table-prompt-v2-short"
)
DOCUMENTS_ROOT = CHANDRA_ROOT / "documents"
FAILURES_ROOT = CHANDRA_ROOT / "failures"
LOCAL_WORK_ROOT = Path("/content/chandra2-table-prompt-v2-work") / EXPERIMENT_ID
for directory in (DOCUMENTS_ROOT, FAILURES_ROOT, LOCAL_WORK_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def write_json(path: Path, payload: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True, default=str) + "\n",
        encoding="utf-8",
    )

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def completion_is_valid(document: dict, document_dir: Path) -> bool:
    marker_path = document_dir / "completion.json"
    if not marker_path.is_file():
        return False
    marker = json.loads(marker_path.read_text(encoding="utf-8"))
    return (
        marker.get("status") == "completed"
        and marker.get("input_sha256") == document["sha256"]
        and marker.get("config_hash") == config_hash
        and (document_dir / "metadata.json").is_file()
        and (document_dir / "result.md").is_file()
        and (document_dir / "result.html").is_file()
        and (document_dir / "raw.html").is_file()
        and (document_dir / "chunks.json").is_file()
    )

def archive_partial_output(document_dir: Path) -> None:
    if not document_dir.exists():
        return
    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archived = document_dir.with_name(f"{document_dir.name}.partial-{suffix}")
    document_dir.rename(archived)
    print(f"Archived incomplete output: {archived}")

In [13]:
selected_documents = [
    document for document in manifest["documents"]
    if document["document_id"] in TARGET_DOCUMENT_IDS
]
if len(selected_documents) != len(TARGET_DOCUMENT_IDS):
    found = {document["document_id"] for document in selected_documents}
    raise RuntimeError(f"Missing target documents: {sorted(TARGET_DOCUMENT_IDS - found)}")
print("Selected:", [document["relative_path"] for document in selected_documents])

run_records = []
for document in selected_documents:
    document_id = document["document_id"]
    input_path = EXTRACT_DIR / "corpus" / document_id / document["filename"]
    drive_document_dir = DOCUMENTS_ROOT / document_id

    if completion_is_valid(document, drive_document_dir):
        print(f"SKIP completed: {document['relative_path']}")
        run_records.append({"document_id": document_id, "status": "skipped"})
        continue

    archive_partial_output(drive_document_dir)
    local_document_dir = LOCAL_WORK_ROOT / f"{document_id}-{int(time.time())}"
    local_document_dir.mkdir(parents=True, exist_ok=False)
    started = time.monotonic()
    try:
        pages = load_file(str(input_path), {})
        if not pages:
            raise RuntimeError("Chandra2 không đọc được trang nào.")

        results = []
        for batch_start in range(0, len(pages), BATCH_SIZE):
            page_batch = pages[batch_start : batch_start + BATCH_SIZE]
            batch = [
                BatchInputItem(image=image, prompt=CUSTOM_PROMPT) for image in page_batch
            ]
            with torch.inference_mode():
                generated = list(
                    manager.generate(
                        batch,
                        include_images=INCLUDE_IMAGES,
                        include_headers_footers=INCLUDE_HEADERS_FOOTERS,
                        max_output_tokens=MAX_OUTPUT_TOKENS,
                    )
                )
            if len(generated) != len(batch):
                raise RuntimeError("Số output không khớp số trang input.")
            failed_pages = [
                batch_start + index + 1
                for index, result in enumerate(generated)
                if result.error
            ]
            if failed_pages:
                raise RuntimeError(f"Model lỗi tại pages: {failed_pages}")
            results.extend(generated)

        pages_dir = local_document_dir / "pages"
        images_dir = local_document_dir / "images"
        pages_dir.mkdir()
        images_dir.mkdir()
        document_pages = []
        label_counts = Counter()
        raw_sections = []
        clean_sections = []
        markdown_sections = []

        for page_number, result in enumerate(results, start=1):
            prefix = f"page_{page_number:04d}"
            raw_html = result.raw or ""
            clean_html = result.html or ""
            page_markdown = result.markdown or ""
            blocks = list(result.chunks or [])
            label_counts.update(str(block.get("label", "block")) for block in blocks)
            page_images_dir = images_dir / prefix
            page_images_dir.mkdir()
            image_files = []
            for image_name, image in (result.images or {}).items():
                image_path = page_images_dir / Path(image_name).name
                image.save(image_path)
                image_files.append(image_path.relative_to(local_document_dir).as_posix())

            page_payload = {
                "page_number": page_number,
                "page_box": list(result.page_box or []),
                "token_count": int(result.token_count or 0),
                "blocks": blocks,
                "image_files": image_files,
            }
            (pages_dir / f"{prefix}.raw.html").write_text(raw_html, encoding="utf-8")
            (pages_dir / f"{prefix}.clean.html").write_text(clean_html, encoding="utf-8")
            (pages_dir / f"{prefix}.md").write_text(page_markdown, encoding="utf-8")
            write_json(pages_dir / f"{prefix}.chunks.json", page_payload)
            raw_sections.append(f'<section data-page-number="{page_number}">\n{raw_html}\n</section>')
            clean_sections.append(f'<section data-page-number="{page_number}">\n{clean_html}\n</section>')
            markdown_sections.append(page_markdown)
            document_pages.append(page_payload)

        latency_seconds = round(time.monotonic() - started, 3)
        token_count = sum(int(result.token_count or 0) for result in results)
        (local_document_dir / "raw.html").write_text("\n\n".join(raw_sections), encoding="utf-8")
        (local_document_dir / "result.html").write_text("\n\n".join(clean_sections), encoding="utf-8")
        (local_document_dir / "result.md").write_text("\n\n".join(markdown_sections), encoding="utf-8")
        write_json(
            local_document_dir / "chunks.json",
            {"document_id": document_id, "input_sha256": document["sha256"], "pages": document_pages},
        )
        metadata = {
            "status": "completed",
            "completed_at": utc_now(),
            "experiment_id": EXPERIMENT_ID,
            "document_id": document_id,
            "filename": document["filename"],
            "relative_path": document["relative_path"],
            "input_sha256": document["sha256"],
            "page_count": len(results),
            "token_count": token_count,
            "latency_seconds": latency_seconds,
            "label_counts": dict(sorted(label_counts.items())),
            "gpu_name": gpu.name,
            "gpu_memory_gib": round(gpu_memory_gib, 3),
            "cuda_version": torch.version.cuda,
            "model_revision": model_revision,
            "prompt_hash": prompt_hash,
            "config_hash": config_hash,
            "generation_config": generation_config,
        }
        write_json(local_document_dir / "metadata.json", metadata)

        shutil.copytree(local_document_dir, drive_document_dir)
        completion = {
            "status": "completed",
            "completed_at": metadata["completed_at"],
            "input_sha256": document["sha256"],
            "config_hash": config_hash,
        }
        write_json(drive_document_dir / "completion.json", completion)
        run_records.append(
            {"document_id": document_id, "status": "completed", "page_count": len(results)}
        )
        print(
            f"DONE {document['relative_path']}: {len(results)} pages, "
            f"{token_count} tokens, {latency_seconds}s"
        )
    except Exception as exc:
        failure = {
            "status": "failed",
            "failed_at": utc_now(),
            "experiment_id": EXPERIMENT_ID,
            "document_id": document_id,
            "relative_path": document["relative_path"],
            "input_sha256": document["sha256"],
            "config_hash": config_hash,
            "error_type": type(exc).__name__,
            "error": str(exc),
        }
        failure_path = FAILURES_ROOT / f"{document_id}-{int(time.time())}.json"
        write_json(failure_path, failure)
        run_records.append({"document_id": document_id, "status": "failed", "error": str(exc)})
        print(f"FAILED {document['relative_path']}: {type(exc).__name__}: {exc}")
    finally:
        write_json(
            CHANDRA_ROOT / "run_summary.json",
            {
                "experiment_id": EXPERIMENT_ID,
                "updated_at": utc_now(),
                "config_hash": config_hash,
                "records": run_records,
            },
        )
        torch.cuda.empty_cache()

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


Selected: ['UET. SƠ ĐỒ TRÍ GIAN TRẠI UET CONNECT 2022.pdf']
DONE UET. SƠ ĐỒ TRÍ GIAN TRẠI UET CONNECT 2022.pdf: 1 pages, 1489 tokens, 93.552s


In [14]:
completed_count = sum(record["status"] in {"completed", "skipped"} for record in run_records)
failed_count = sum(record["status"] == "failed" for record in run_records)
print(f"Finished: {completed_count}/5 available, failures={failed_count}")
print(f"Outputs: {CHANDRA_ROOT}")
subprocess.run(["nvidia-smi"], check=True)
if failed_count:
    raise RuntimeError("Có document bị lỗi. Xem chandra2/failures và chạy lại notebook để resume.")

Finished: 1/5 available, failures=0
Outputs: /content/drive/MyDrive/AXIOM_DE-RD/parser-comparison/20260722T155731-parser-smoke-571c3721/chandra2-table-prompt-v2-short


In [15]:
# Second pass: crop every detected Table block and OCR it independently.
import math
import re

CROP_VARIANT = "chandra2-table-crop-v1"
CROP_MARGIN_RATIO = 0.02
CROP_MIN_SHORT_SIDE = 1536
CROP_MAX_LONG_SIDE = 3072
CROP_MAX_PIXELS = 3072 * 2048
CROP_MAX_OUTPUT_TOKENS = 4096

TABLE_CROP_PROMPT = """
OCR this image as exactly one HTML table.
Return only one complete <table>...</table>; no prose, markdown, JSON, or outer div.
First reconstruct the complete logical row and column grid from the visible borders.
A missing internal horizontal border means the cell spans rows; encode it with rowspan.
A missing internal vertical border means the cell spans columns; encode it with colspan.
Place each merged cell at its upper-left grid position and emit its content exactly once.
Do not emit td/th elements for grid positions covered by a previous rowspan or colspan.
Do not assign vertically centered text to the middle row of a merged cell.
Preserve visible text and genuinely empty cells exactly.
The expanded span grid must be rectangular and non-overlapping.
Use only table, thead, tbody, tr, th, td, b, i, br, span, sup, sub, and math tags.
""".strip()

crop_prompt_hash = hashlib.sha256(TABLE_CROP_PROMPT.encode("utf-8")).hexdigest()
crop_config = {
    "variant": CROP_VARIANT,
    "model_checkpoint": MODEL_CHECKPOINT,
    "model_revision": model_revision,
    "prompt_hash": crop_prompt_hash,
    "margin_ratio": CROP_MARGIN_RATIO,
    "min_short_side": CROP_MIN_SHORT_SIDE,
    "max_long_side": CROP_MAX_LONG_SIDE,
    "max_pixels": CROP_MAX_PIXELS,
    "max_output_tokens": CROP_MAX_OUTPUT_TOKENS,
    "include_images": False,
    "include_headers_footers": True,
}
crop_config_hash = hashlib.sha256(
    json.dumps(crop_config, sort_keys=True).encode("utf-8")
).hexdigest()
CROP_ROOT = DRIVE_EXPERIMENT_DIR / CROP_VARIANT
CROP_DOCUMENTS_ROOT = CROP_ROOT / "documents"
CROP_DOCUMENTS_ROOT.mkdir(parents=True, exist_ok=True)

def extract_first_table(html: str) -> str:
    match = re.search(r"<table\b[^>]*>.*?</table>", html or "", flags=re.I | re.S)
    return match.group(0).strip() if match else (html or "").strip()

def span_count(html: str, attribute: str) -> int:
    return len(re.findall(rf"\b{attribute}\s*=", html or "", flags=re.I))

def crop_box_from_bbox(image, bbox, page_box):
    if len(bbox) != 4:
        raise ValueError(f"Invalid table bbox: {bbox}")
    if len(page_box) == 4:
        px0, py0, px1, py1 = [float(value) for value in page_box]
    else:
        px0, py0, px1, py1 = 0.0, 0.0, float(image.width), float(image.height)
    if px1 <= px0 or py1 <= py0:
        raise ValueError(f"Invalid page box: {page_box}")
    x0, y0, x1, y1 = [float(value) for value in bbox]
    left = (x0 - px0) / (px1 - px0) * image.width
    top = (y0 - py0) / (py1 - py0) * image.height
    right = (x1 - px0) / (px1 - px0) * image.width
    bottom = (y1 - py0) / (py1 - py0) * image.height
    margin = max(right - left, bottom - top) * CROP_MARGIN_RATIO
    return (
        max(0, math.floor(left - margin)),
        max(0, math.floor(top - margin)),
        min(image.width, math.ceil(right + margin)),
        min(image.height, math.ceil(bottom + margin)),
    )

def resize_crop_for_model(image):
    short_side = min(image.size)
    long_side = max(image.size)
    scale = min(
        CROP_MIN_SHORT_SIDE / short_side,
        CROP_MAX_LONG_SIDE / long_side,
        math.sqrt(CROP_MAX_PIXELS / (image.width * image.height)),
    )
    if scale <= 1:
        return image.copy()
    new_size = (round(image.width * scale), round(image.height * scale))
    return image.resize(new_size, resample=3)  # PIL.Image.Resampling.LANCZOS

crop_run_records = []
for document in selected_documents:
    document_id = document["document_id"]
    input_path = EXTRACT_DIR / "corpus" / document_id / document["filename"]
    source_document_dir = DOCUMENTS_ROOT / document_id
    chunks_path = source_document_dir / "chunks.json"
    if not chunks_path.is_file():
        raise FileNotFoundError(
            f"Missing {chunks_path}. Run the full-page cell above once before crop pass."
        )

    source_chunks = json.loads(chunks_path.read_text(encoding="utf-8"))
    crop_pages = load_file(str(input_path), {})
    if len(crop_pages) != len(source_chunks.get("pages", [])):
        raise RuntimeError("Page count mismatch between source file and chunks.json.")

    local_crop_dir = Path("/content/chandra2-table-crop-work") / EXPERIMENT_ID / document_id
    if local_crop_dir.exists():
        shutil.rmtree(local_crop_dir)
    table_crops_dir = local_crop_dir / "table_crops"
    table_crops_dir.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    table_records = []
    replacement_tables = []

    for page_index, (page_image, page_data) in enumerate(
        zip(crop_pages, source_chunks["pages"]), start=1
    ):
        table_blocks = [
            block for block in page_data.get("blocks", [])
            if str(block.get("label", "")).lower() == "table"
        ]
        for table_index, block in enumerate(table_blocks, start=1):
            crop_id = f"page_{page_index:04d}_table_{table_index:02d}"
            crop_box = crop_box_from_bbox(
                page_image, block.get("bbox", []), page_data.get("page_box", [])
            )
            original_crop = page_image.crop(crop_box)
            model_crop = resize_crop_for_model(original_crop)
            original_crop.save(table_crops_dir / f"{crop_id}.original.png")
            model_crop.save(table_crops_dir / f"{crop_id}.model-input.png")

            with torch.inference_mode():
                generated = list(
                    manager.generate(
                        [BatchInputItem(image=model_crop, prompt=TABLE_CROP_PROMPT)],
                        include_images=False,
                        include_headers_footers=True,
                        max_output_tokens=CROP_MAX_OUTPUT_TOKENS,
                    )
                )
            if len(generated) != 1 or generated[0].error:
                error = generated[0].error if generated else "missing output"
                raise RuntimeError(f"Crop {crop_id} failed: {error}")

            crop_result = generated[0]
            raw_html = crop_result.raw or ""
            clean_html = crop_result.html or ""
            table_html = extract_first_table(clean_html or raw_html)
            (table_crops_dir / f"{crop_id}.raw.html").write_text(raw_html, encoding="utf-8")
            (table_crops_dir / f"{crop_id}.clean.html").write_text(clean_html, encoding="utf-8")
            (table_crops_dir / f"{crop_id}.table.html").write_text(table_html, encoding="utf-8")
            rowspans = span_count(table_html, "rowspan")
            colspans = span_count(table_html, "colspan")
            record = {
                "crop_id": crop_id,
                "page_number": page_index,
                "table_index": table_index,
                "source_bbox": list(block.get("bbox", [])),
                "crop_box_pixels": list(crop_box),
                "original_size": list(original_crop.size),
                "model_input_size": list(model_crop.size),
                "token_count": int(crop_result.token_count or 0),
                "rowspan_count": rowspans,
                "colspan_count": colspans,
                "table_file": f"table_crops/{crop_id}.table.html",
            }
            table_records.append(record)
            replacement_tables.append(table_html)
            print(
                f"CROP {crop_id}: bbox={record['source_bbox']}, "
                f"input={model_crop.width}x{model_crop.height}, "
                f"rowspan={rowspans}, colspan={colspans}"
            )

    if not table_records:
        raise RuntimeError("No Table blocks found in source chunks.json.")

    source_html_path = source_document_dir / "result.html"
    source_html = source_html_path.read_text(encoding="utf-8")
    replacement_iterator = iter(replacement_tables)
    enriched_html, replaced_count = re.subn(
        r"<table\b[^>]*>.*?</table>",
        lambda _: next(replacement_iterator),
        source_html,
        count=len(replacement_tables),
        flags=re.I | re.S,
    )
    if replaced_count != len(replacement_tables):
        raise RuntimeError(
            f"Detected {len(replacement_tables)} table crops but replaced {replaced_count}."
        )

    (local_crop_dir / "prompt.txt").write_text(TABLE_CROP_PROMPT + "\n", encoding="utf-8")
    (local_crop_dir / "full-page.result.html").write_text(source_html, encoding="utf-8")
    (local_crop_dir / "result.tables.html").write_text("\n\n".join(replacement_tables), encoding="utf-8")
    (local_crop_dir / "result.crop-enriched.html").write_text(enriched_html, encoding="utf-8")
    write_json(local_crop_dir / "tables.json", {"tables": table_records})
    latency_seconds = round(time.monotonic() - started, 3)
    metadata = {
        "status": "completed",
        "completed_at": utc_now(),
        "experiment_id": EXPERIMENT_ID,
        "document_id": document_id,
        "input_sha256": document["sha256"],
        "source_full_page_dir": str(source_document_dir),
        "config_hash": crop_config_hash,
        "generation_config": crop_config,
        "table_count": len(table_records),
        "rowspan_count": sum(item["rowspan_count"] for item in table_records),
        "colspan_count": sum(item["colspan_count"] for item in table_records),
        "token_count": sum(item["token_count"] for item in table_records),
        "latency_seconds": latency_seconds,
        "gpu_name": gpu.name,
    }
    write_json(local_crop_dir / "metadata.json", metadata)
    write_json(
        local_crop_dir / "completion.json",
        {
            "status": "completed",
            "completed_at": metadata["completed_at"],
            "input_sha256": document["sha256"],
            "config_hash": crop_config_hash,
        },
    )

    drive_crop_dir = CROP_DOCUMENTS_ROOT / document_id
    if drive_crop_dir.exists():
        archived = drive_crop_dir.with_name(
            f"{document_id}.previous-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
        )
        drive_crop_dir.rename(archived)
    shutil.copytree(local_crop_dir, drive_crop_dir)
    crop_run_records.append({"document_id": document_id, **metadata})
    write_json(CROP_ROOT / "run_summary.json", {"records": crop_run_records})
    print(
        f"DONE crop pass: tables={metadata['table_count']}, "
        f"rowspan={metadata['rowspan_count']}, colspan={metadata['colspan_count']}, "
        f"output={drive_crop_dir}"
    )
    torch.cuda.empty_cache()

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


CROP page_0001_table_01: bbox=[100, 1437, 743, 1830], input=2452x1536, rowspan=0, colspan=0


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`


CROP page_0001_table_02: bbox=[805, 1181, 1462, 2070], input=1536x2050, rowspan=6, colspan=0
DONE crop pass: tables=2, rowspan=6, colspan=0, output=/content/drive/MyDrive/AXIOM_DE-RD/parser-comparison/20260722T155731-parser-smoke-571c3721/chandra2-table-crop-v1/documents/95168ff83bfc14d8
